[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AllInVaders/aistudio-full-course/blob/main/notebooks/03_Gemini_Live_API_Tool_Agents_and_Antigravity_SDK.ipynb)

# Module 03: Gemini Live API Bidirectional Streaming, Tool Agents & Antigravity Orchestration
### Módulo 03: Gemini Live API en Tiempo Real, Agentes con Herramientas y Orquestación Antigravity

**English Overview**: Build **Stage 2 of the Flagship Project (Live Multimodal Copilot + Tool Agent)**. Connect to `gemini-2.0-flash-live-001` over async bidirectional streaming, invoke real-time Python tools (`calculate_unit_economics`, `check_inventory_status`), and orchestrate multi-step specialist subagents.

**Resumen en Español**: Construye la **Etapa 2 del Proyecto Insignia (Copiloto Multimodal en Vivo + Agente con Herramientas)**. Conéctate a `gemini-2.0-flash-live-001` mediante streaming bidireccional asíncrono, ejecuta herramientas Python en tiempo real y orquesta subagentes especialistas.

In [ ]:
%pip install -q -U google-genai pydantic nest_asyncio

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from google import genai
from google.genai import types

def calculate_unit_economics(unit_cost_usd: float, retail_price_usd: float, cac_usd: float) -> dict:
    """Calculates gross margin percentage and net contribution margin per unit."""
    gross_margin = retail_price_usd - unit_cost_usd
    gross_pct = round((gross_margin / retail_price_usd) * 100.0, 2) if retail_price_usd > 0 else 0.0
    net_contribution = round(gross_margin - cac_usd, 2)
    return {
        'gross_margin_pct': gross_pct,
        'net_contribution_usd': net_contribution,
        'verdict': 'HEALTHY' if net_contribution >= 15.0 else 'TIGHT_MARGIN',
    }

TOOLS = {'calculate_unit_economics': calculate_unit_economics}
client = genai.Client()

In [ ]:
async def demo_live_copilot():
    config = types.LiveConnectConfig(
        response_modalities=['TEXT'],
        tools=[calculate_unit_economics],
    )
    async with client.aio.live.connect(model='gemini-2.0-flash-live-001', config=config) as session:
        prompt = 'Calculate unit economics for unit cost $28, retail price $99, and CAC $24.'
        await session.send_client_content(
            turns=types.Content(role='user', parts=[types.Part.from_text(text=prompt)]),
            turn_complete=True,
        )
        async for msg in session.receive():
            if msg.tool_call:
                responses = []
                for fc in msg.tool_call.function_calls:
                    out = TOOLS[fc.name](**fc.args)
                    print(f'[Tool Executed] {fc.name} -> {out}')
                    responses.append(types.FunctionResponse(id=fc.id, name=fc.name, response=out))
                await session.send_tool_response(function_responses=responses)
            if msg.server_content and msg.server_content.model_turn:
                for part in msg.server_content.model_turn.parts:
                    if part.text:
                        print(part.text, end='')
            if msg.server_content and msg.server_content.turn_complete:
                print()
                break

await demo_live_copilot()